In [3]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np

Goal: To understand what happens with One-Hot Encoding in Attention

In [52]:
def Attention(Q, K, V):
    '''
    A simple attention mechanism
    '''
    assert len(Q.shape) == len(K.shape) == len(V.shape) == 3, "Invalid Dims for Tensors"
    
    batch, seq_len, d_k = Q.shape # dim is last in mlx

    # Association between Q and K: [Batch, N, Dim] @ [Batch, Dim, N] --> [Batch, N, N]
    attention = (Q @ K.reshape(batch, d_k, seq_len)) / (d_k**0.5)


    # Softmax on Association values: [Batch, N, N]
    attention = mx.softmax(attention, axis = -1)

    # Weighted Average Values
    return attention, attention @ V

    


In [53]:
def one_hot(num_classes: int, seq_len: int):
    """
    Function to create a random one hot vector
    """
    output_tensor = [[0 for _ in range(num_classes)] for _ in range(seq_len)]
    for idx in range(seq_len):
        random_class = mx.random.randint(low = 0, high = num_classes).item()
        output_tensor[idx][random_class] = 1
    return mx.array(output_tensor)
    


In [54]:
VOCAB_SIZE = 3
SEQ_LEN = 10

In [55]:
one_hot_input = one_hot(VOCAB_SIZE, SEQ_LEN)
print(f"Shape of one hot: {one_hot_input.shape} \n")
print(f"{one_hot_input}")
mx.eval(one_hot_input) # Default is lazy init

Shape of one hot: (10, 3) 

array([[0, 1, 0],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [1, 0, 0],
       [1, 0, 0],
       [0, 1, 0]], dtype=int32)


In [56]:
# Q, K and V transformation
Q_linear = nn.Linear(input_dims = VOCAB_SIZE, output_dims = VOCAB_SIZE - 1) # For easy visualization
K_linear = nn.Linear(input_dims = VOCAB_SIZE, output_dims = VOCAB_SIZE - 1) # For easy visualization
V_linear = nn.Linear(input_dims = VOCAB_SIZE, output_dims = VOCAB_SIZE - 1) # For easy visualization

# Init layers
mx.eval(Q_linear); mx.eval(K_linear); mx.eval(V_linear)

# By default, no graphs
Q, K, V = Q_linear(one_hot_input), K_linear(one_hot_input), V_linear(one_hot_input)

# View the tensors:
print(f"Q Vector: {Q.shape} \n {Q} \n K Vector: {K.shape} \n {K} \n V Vector: {V.shape} \n {V}")

Q Vector: (10, 2) 
 array([[-0.202714, -0.507799],
       [0.215, -0.402715],
       [0.215, -0.402715],
       ...,
       [0.409442, -0.603005],
       [0.409442, -0.603005],
       [-0.202714, -0.507799]], dtype=float32) 
 K Vector: (10, 2) 
 array([[-0.192155, 0.146232],
       [0.125268, 0.370292],
       [0.125268, 0.370292],
       ...,
       [-0.144883, 0.514584],
       [-0.144883, 0.514584],
       [-0.192155, 0.146232]], dtype=float32) 
 V Vector: (10, 2) 
 array([[-0.848987, 0.540707],
       [-0.650815, 0.142339],
       [-0.650815, 0.142339],
       ...,
       [-0.17544, 0.445851],
       [-0.17544, 0.445851],
       [-0.848987, 0.540707]], dtype=float32)


In [57]:
Q, K, V = mx.expand_dims(Q, 0), mx.expand_dims(K, 0), mx.expand_dims(V, 0)
attention, value = Attention(Q, K, V)

## Token → Q, K, V Mapping (seed=42, VOCAB_SIZE=3, SEQ_LEN=10)

> Each one-hot vector selects a single row from the weight matrix — so tokens of the same class get **identical** Q, K, and V vectors.

| Token | Class | One-Hot | Q | K | V |
|-------|-------|---------|---|---|---|
| t0 | c2 | [0, 0, 1] | [+0.3115, -0.2798] | [-0.9876, +0.7905] | [+0.0607, -0.5529] |
| t1 | c1 | [0, 1, 0] | [+0.6229, -0.3310] | [-0.9053, -0.0104] | [+0.5525, +0.3043] |
| t2 | c0 | [1, 0, 0] | [+0.8971, -0.3601] | [+0.0549, +0.0708] | [+0.9215, -0.0874] |
| t3 | c2 | [0, 0, 1] | [+0.3115, -0.2798] | [-0.9876, +0.7905] | [+0.0607, -0.5529] |
| t4 | c1 | [0, 1, 0] | [+0.6229, -0.3310] | [-0.9053, -0.0104] | [+0.5525, +0.3043] |
| t5 | c0 | [1, 0, 0] | [+0.8971, -0.3601] | [+0.0549, +0.0708] | [+0.9215, -0.0874] |
| t6 | c0 | [1, 0, 0] | [+0.8971, -0.3601] | [+0.0549, +0.0708] | [+0.9215, -0.0874] |
| t7 | c2 | [0, 0, 1] | [+0.3115, -0.2798] | [-0.9876, +0.7905] | [+0.0607, -0.5529] |
| t8 | c1 | [0, 1, 0] | [+0.6229, -0.3310] | [-0.9053, -0.0104] | [+0.5525, +0.3043] |
| t9 | c2 | [0, 0, 1] | [+0.3115, -0.2798] | [-0.9876, +0.7905] | [+0.0607, -0.5529] |

When you use one hot encoding, the Q, K and V vectors are the same for a class, regardless of where it occurs -- this is corrected with positional encoding

Also, if certain classes are closer to another we lose that semantic structure. The weight matrix has nothing to build on. 

## Attention Weights (Query × Key)

> Rows = query token, Columns = key token. Notice that **identical classes produce identical attention rows** — the model cannot distinguish t0 from t3, t7, or t9.

| Query \ Key | t0 (c2) | t1 (c1) | t2 (c0) | t3 (c2) | t4 (c1) | t5 (c0) | t6 (c0) | t7 (c2) | t8 (c1) | t9 (c2) |
|---|---|---|---|---|---|---|---|---|---|---|
| **t0 (c2)** | 0.0807 | 0.1189 | 0.0821 | 0.0997 | 0.1247 | 0.0880 | 0.0975 | 0.1209 | 0.1009 | 0.0865 |
| **t1 (c1)** | 0.0658 | 0.1434 | 0.0682 | 0.1008 | 0.1329 | 0.0883 | 0.0823 | 0.1462 | 0.0870 | 0.0852 |
| **t2 (c0)** | 0.0542 | 0.1666 | 0.0571 | 0.1003 | 0.1368 | 0.0879 | 0.0692 | 0.1701 | 0.0744 | 0.0835 |
| **t3 (c2)** | 0.0807 | 0.1189 | 0.0821 | 0.0997 | 0.1247 | 0.0880 | 0.0975 | 0.1209 | 0.1009 | 0.0865 |
| **t4 (c1)** | 0.0658 | 0.1434 | 0.0682 | 0.1008 | 0.1329 | 0.0883 | 0.0823 | 0.1462 | 0.0870 | 0.0852 |
| **t5 (c0)** | 0.0542 | 0.1666 | 0.0571 | 0.1003 | 0.1368 | 0.0879 | 0.0692 | 0.1701 | 0.0744 | 0.0835 |
| **t6 (c0)** | 0.0542 | 0.1666 | 0.0571 | 0.1003 | 0.1368 | 0.0879 | 0.0692 | 0.1701 | 0.0744 | 0.0835 |
| **t7 (c2)** | 0.0807 | 0.1189 | 0.0821 | 0.0997 | 0.1247 | 0.0880 | 0.0975 | 0.1209 | 0.1009 | 0.0865 |
| **t8 (c1)** | 0.0658 | 0.1434 | 0.0682 | 0.1008 | 0.1329 | 0.0883 | 0.0823 | 0.1462 | 0.0870 | 0.0852 |
| **t9 (c2)** | 0.0807 | 0.1189 | 0.0821 | 0.0997 | 0.1247 | 0.0880 | 0.0975 | 0.1209 | 0.1009 | 0.0865 |

Coming to the attention weights, we can decompose the above matrix to:

c0 -> c0; c0 -> c1; c0 -> c2

c1 -> c0; c1 -> c1; c1 -> c2

c2 -> c0; c2 -> c1; c2 -> c2